In [1]:
import pandas as pd

roi_q = pd.read_csv("roi_quarterly.csv")
roi_inf = pd.read_csv("roi_inflation_quarterly.csv")

roi_merged = roi_q.merge(
    roi_inf,
    on=["Year", "Quarter", "Period"],
    how="left"
)


In [2]:
print(roi_merged.head())
print("Missing inflation:",
      roi_merged["Inflation_Rate"].isna().mean())


   County  Year  Quarter   Period  transaction_count      avg_price  \
0  Carlow  2010        1  2010 Q1               40.0  171800.524750   
1  Carlow  2010        2  2010 Q2               44.0  179575.795455   
2  Carlow  2010        3  2010 Q3               63.0  179927.914286   
3  Carlow  2010        4  2010 Q4               84.0  168287.516310   
4  Carlow  2011        1  2011 Q1               41.0  164626.936341   

   Inflation_Rate  
0           -3.37  
1           -1.33  
2            0.20  
3            0.93  
4            2.27  
Missing inflation: 0.0


In [3]:
roi_merged["real_avg_price"] = (
    roi_merged["avg_price"] /
    (1 + roi_merged["Inflation_Rate"] / 100)
).round(2)


In [4]:
roi_merged["avg_price"] = roi_merged["avg_price"].round(2)
roi_merged["Inflation_Rate"] = roi_merged["Inflation_Rate"].round(2)


In [5]:
roi_merged.to_csv("roi_quarterly_inflation.csv", index=False)


In [6]:
ni = pd.read_csv("ni_quarterly_inflation.csv")
roi = pd.read_csv("roi_quarterly_inflation.csv")

# Ensure consistent types
for df in [ni, roi]:
    df["Year"] = df["Year"].astype(int)
    df["Quarter"] = df["Quarter"].astype(int)
    df["avg_price"] = pd.to_numeric(df["avg_price"], errors="coerce")
    df["Inflation_Rate"] = pd.to_numeric(df["Inflation_Rate"], errors="coerce")

# Combine
all_island = pd.concat([roi, ni], ignore_index=True)

# Sort nicely
all_island = all_island.sort_values(
    ["County", "Year", "Quarter"]
).reset_index(drop=True)

# Save
all_island.to_csv("all_island_quarterly_inflation.csv", index=False)

print(all_island.head())

                    County  Year  Quarter   Period  transaction_count  \
0  Antrim and Newtownabbey  2010        1  2010 Q1              217.0   
1  Antrim and Newtownabbey  2010        2  2010 Q2              245.0   
2  Antrim and Newtownabbey  2010        3  2010 Q3              269.0   
3  Antrim and Newtownabbey  2010        4  2010 Q4              270.0   
4  Antrim and Newtownabbey  2011        1  2011 Q1              205.0   

   avg_price  Inflation_Rate  real_avg_price  
0  148341.74            3.28       143630.65  
1  152846.09            3.46       147734.48  
2  156920.99            3.09       152217.47  
3  138840.47            3.38       134301.09  
4  134269.20            4.12       128956.20  


In [7]:
import plotly.express as px

county = "Dublin"

df = all_island[all_island["County"] == county]

fig = px.line(
    df,
    x="Period",
    y="avg_price",
    markers=True,
    title=f"Average Property Price Trend – {county}",
    labels={"avg_price": "Average Price (€)"}
)

fig.show()


In [8]:
counties = ["Dublin", "Belfast", "Cork", "Antrim And Newtownabbey"]

df = all_island[all_island["County"].isin(counties)]

fig = px.line(
    df,
    x="Period",
    y="avg_price",
    color="County",
    title="Quarterly House Price Trends (Selected Regions)",
    labels={"avg_price": "Average Price (€)"}
)

fig.show()


In [9]:
fig = px.line(
    df,
    x="Period",
    y=["avg_price", "Inflation_Rate"],
    title="House Prices vs Inflation",
)

fig.show()


In [11]:
df = pd.read_csv("all_island_quarterly_inflation.csv")

ni_counties = {
    "Antrim and Newtownabbey",
    "Ards and North Down",
    "Armagh City, Banbridge and Craigavon",
    "Belfast",
    "Causeway Coast and Glens",
    "Derry City and Strabane",
    "Fermanagh and Omagh",
    "Lisburn and Castlereagh",
    "Mid Ulster",
    "Mid and East Antrim",
    "Newry, Mourne and Down"
}

df["Region"] = df["County"].apply(
    lambda x: "Northern Ireland" if x in ni_counties else "Republic of Ireland"
)

df.to_csv("all_island_quarterly_inflation.csv", index=False)

print(df[["County", "Region"]].drop_duplicates())


                                    County               Region
0                  Antrim and Newtownabbey     Northern Ireland
63                     Ards and North Down     Northern Ireland
126   Armagh City, Banbridge and Craigavon     Northern Ireland
189                                Belfast     Northern Ireland
252                                 Carlow  Republic of Ireland
316               Causeway Coast and Glens     Northern Ireland
379                                  Cavan  Republic of Ireland
443                                  Clare  Republic of Ireland
507                                   Cork  Republic of Ireland
571                Derry City and Strabane     Northern Ireland
634                                Donegal  Republic of Ireland
698                                 Dublin  Republic of Ireland
762                    Fermanagh and Omagh     Northern Ireland
825                                 Galway  Republic of Ireland
889                                  Ker